In [11]:
# !unzip -q -o ./data/sirius-nlp-contest.zip -d ./data/
# !tar -xzvf ./models/baai-transformers-bge-small-en-v1.5-v1.tar.gz -C ./models/

In [37]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

## Dependencies

In [68]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

import torch

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

import matplotlib.pyplot as plt
import seaborn as sns

import re
import emoji

from tqdm.cli import tqdm
from tqdm import trange

tqdm.pandas()

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('names', quiet=True)

True

## EDA

In [77]:
df_train = pd.read_csv("./data/train.csv")
df_test_doc = pd.read_csv('./data/test-docs.csv')
df_test_questions = pd.read_csv('./data/test-questions.csv')
df_train.head()

,question,context,c_id
0,To whom did the Virgin Mary allegedly appear i...,"Architecturally, the school has a Catholic cha...",0
1,What is in front of the Notre Dame Main Building?,"Architecturally, the school has a Catholic cha...",0
2,The Basilica of the Sacred heart at Notre Dame...,"Architecturally, the school has a Catholic cha...",0
3,What is the Grotto at Notre Dame?,"Architecturally, the school has a Catholic cha...",0
4,What sits on top of the Main Building at Notre...,"Architecturally, the school has a Catholic cha...",0


In [78]:
df_train.shape

(87599, 3)

In [79]:
df_train['question'].nunique()

87355

In [80]:
df_train['context'].nunique()

18891

In [81]:
df_train['c_id'].describe()

count    87599.000000
mean      9477.616000
std       5495.417059
min          0.000000
25%       4903.000000
50%       9458.000000
75%      14245.000000
max      18890.000000
Name: c_id, dtype: float64

In [82]:
df_test_doc.head()

,context
0,China is the country with the largest populati...
1,The bulk of remaining commercial flight offeri...
2,Based on the similar shifts underway the natio...
3,"Since 2004, through the V&A + RIBA Architectur..."
4,"Society in the Japanese ""Tokugawa period"" (Edo..."


In [83]:
train_context = set(df_train['context'].to_list())
test_context = set(df_test_doc['context'].to_list())

len(train_context), len(test_context)

(18891, 18891)

In [84]:
len(train_context & test_context)

18891

In [85]:
id_to_context = df_test_doc['context'].to_dict()
context_to_id = {v: k for k, v in id_to_context.items()}

In [86]:
documents = df_train[['context', 'c_id']].drop_duplicates().reset_index(drop=True)
documents.head()

,context,c_id
0,"Architecturally, the school has a Catholic cha...",0
1,"As at most other universities, Notre Dame's st...",1
2,The university is the major seat of the Congre...,2
3,The College of Engineering was established in ...,3
4,All of Notre Dame's undergraduate students are...,4


## Text preparing

In [87]:
def text_normalize(text: str) -> str:
    text = re.sub(r'https?://\S+|www\.\S+', ' __URL__ ', text)
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', ' __EMAIL__ ', text)
    text = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', ' __IP__ ', text)
    text = re.sub(r'[\+]?[(]?[0-9]{1,4}[)]?[-\s\./0-9]{7,}', ' __PHONE__ ', text)
    text = re.sub(r'\b\d{1,2}[/\-\.]\d{1,2}[/\-\.]\d{2,4}\b', ' __DATE__ ', text)
    text = re.sub(r'\b\d{1,2}:\d{2}(?::\d{2})?\s*(?:am|pm)?\b', ' __TIME__ ', text, flags=re.I)
    text = re.sub(r'-?\b\d+\.?\d*\b', ' __NUMBER__ ', text)
    text = re.sub(r'@\w+', ' __USER__ ', text)
    text = re.sub(r'#(\w+)', r' __HASHTAG__ \1 ', text)

    text = text.lower()
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    text = re.sub(r'[^\w\s]', '', text)

    words = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    words = [w for w in words if w not in stop_words or w.startswith('__')]

    # stemmer = PorterStemmer()
    # words = [w if w.startswith('__') and w.endswith('__') else stemmer.stem(w) for w in words]

    lemmatizer = WordNetLemmatizer()
    words = [w if w.startswith('__') and w.endswith('__') else lemmatizer.lemmatize(w) for w in words]

    text = ' '.join(words)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [88]:
clean_documents = documents.copy()
clean_documents['context'] = clean_documents['context'].progress_apply(text_normalize)
clean_documents.head()

100%|██████████| 18891/18891 [00:08<00:00, 2283.73it/s]


,context,c_id
0,architecturally school catholic character atop...,0
1,university notre dame student run number news ...,1
2,university major seat congregation holy cross ...,2
3,college engineering established __number__ how...,3
4,notre dame undergraduate student part one five...,4


## TF-IDF

In [90]:
vectorizer = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1, 3),
    analyzer='word'
)

vectorizer.fit(clean_documents['context'])

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [92]:
X_test = vectorizer.transform(df_test_doc['context'])
Y_test = vectorizer.transform(df_test_questions['question'])

In [96]:
similarity = (X_test @ Y_test.T).toarray()

In [97]:
pred_ids = []
for q_idx, question in tqdm(enumerate(df_test_questions.question), total=df_test_questions.shape[0]):
    best_idx = np.argmax(similarity[:, q_idx]).item()

    pred_ids.append(best_idx)


100%|██████████| 37782/37782 [00:01<00:00, 27049.85it/s]


In [98]:
submission = pd.DataFrame({'id': df_test_questions.index.tolist(), "pred_id": pred_ids})
submission.to_csv("sample_submission.csv", index=False)


## Embeding model